# PTCG Merged Agent Workbench

Parent notebook for **The Pokémon Company - PTCG AI Battle Challenge Simulation**.

## Document map

| Doc | Notebook in `docs/resources/reference_notebooks/` | Role in this workbench |
|-----|-----------------------------------------------------|-------------------------|
| **4/8 — Dragapult** | `a-sample-rule-based-agent-dragapult-ex-deck.ipynb` | **Base policy skeleton** |
| **9/11 — Meta snapshot** | `pok-mon-tcg-ai-battle-meta-snapshot-07-july.ipynb` (+ June 29 sibling on Kaggle) | **Deck choice + holdout mindset** only |
| **10 — Expectimax** | `improved-probabilistic-agent.ipynb` | **Search API + UCB1 + opponent reads** |

## Concrete merge plan

1. **Policy skeleton (Doc 4/8):** keep Dragapult scoring framework — log tracking, deck reconstruction, combo planning, contextual scoring → `DragapultPolicy`.
2. **Deck (Docs 9/11):** replace the default Dragapult list with a **meta-informed** choice — typically **Starmie** or **Festival Thwackey** (underexplored on ladder). Put the 60-card list in `data/deck.csv`.
3. **Search layer (Doc 10):** plug Expectimax **Search API + UCB1** on top of `DragapultPolicy`. Do **not** use Expectimax's `AdvancedPolicy` (that is a Lucario heuristic clone).

## Why this combination

| Source | Strength | Blind spot |
|--------|----------|------------|
| Dragapult | Best public policy architecture | Zero search |
| Expectimax | Real lookahead (UCB1 + Search API) | Weak Lucario base policy |
| Meta author | Ladder analysis + holdout gates | Agent code hidden in `main_b64` / `deck_b64` |

**Target submission:** solid policy + real lookahead + meta-informed deck choice.

> Run cells top-to-bottom. Add `data/deck.csv` (and later `data/cg/`) before packaging.


## 1. Environment setup


In [ ]:
from pathlib import Path

REPO_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
DATA_DIR = REPO_ROOT / "data"
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
REF_DIR = REPO_ROOT / "docs" / "resources" / "reference_notebooks"

SOURCES = {
    "dragapult (Doc 4/8)": REF_DIR / "a-sample-rule-based-agent-dragapult-ex-deck.ipynb",
    "expectimax (Doc 10)": REF_DIR / "improved-probabilistic-agent.ipynb",
    "meta snapshot (Doc 9/11)": REF_DIR / "pok-mon-tcg-ai-battle-meta-snapshot-07-july.ipynb",
}

for path in (DATA_DIR, NOTEBOOKS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Repo:", REPO_ROOT)
print("Data:", DATA_DIR)
print("Reference notebooks:")
for label, path in SOURCES.items():
    print(f"  {'OK' if path.exists() else 'MISSING'} - {label}: {path.name}")


## 2. Meta-informed deck choice (Docs 9/11)

The meta snapshot author publishes **analysis and packaging**, not readable agent code. In their notebooks, submission payloads live as base64 blobs (`profile['main_b64']`, `profile['deck_b64']`).

What we **do** import from them:

- Pick a deck that is **underexplored** but still viable (Starmie ~14% share, Festival Thwackey ~1%).
- Stress-test against ladder pillars before submitting.
- Use **holdout** gates (`holdout_pass` / `holdout_fail`, `PROMOTE_CANDIDATE` / `HOLD_DO_NOT_SUBMIT`) — not raw live-ladder noise.

What we **do not** import:

- Hidden policy logic from base64 payloads.


In [ ]:
import pandas as pd

META_FIELD = pd.DataFrame([
    {"archetype": "starmie", "usage_pct": 13.85, "score_pct": 51.89, "role": "Primary meta candidate - tempo shell, mandatory stress test"},
    {"archetype": "festival_thwackey", "usage_pct": 1.16, "score_pct": 45.99, "role": "Primary meta candidate - low share dark horse"},
    {"archetype": "alakazam_dunsparce", "usage_pct": 18.96, "score_pct": 51.34, "role": "High-share field pillar"},
    {"archetype": "lucario", "usage_pct": 18.78, "score_pct": 42.44, "role": "Crowded - Expectimax baseline policy archetype"},
    {"archetype": "hop_trevenant", "usage_pct": 17.52, "score_pct": 45.47, "role": "Mandatory stress test"},
    {"archetype": "archaludon", "usage_pct": 14.57, "score_pct": 62.20, "role": "Top converter, heavily targeted"},
    {"archetype": "dragapult", "usage_pct": 7.31, "score_pct": 49.13, "role": "Policy skeleton source (default builder constants)"},
])

DECK_CANDIDATES = META_FIELD[META_FIELD["archetype"].isin(["starmie", "festival_thwackey"])]
DECK_CANDIDATES.sort_values("usage_pct")[["archetype", "usage_pct", "score_pct", "role"]]


### Deck workflow

1. Choose **Starmie** or **Festival** from competition data and write 60 card IDs to `data/deck.csv`.
2. The automated builder still emits **Dragapult card constants** today — if your deck is not Dragapult, update those constants in `merged_agent_main.py` (or fork the policy) before submitting.
3. Re-run **section 4 (Build)** and **section 5 (Holdout)** after any deck or policy change.


In [ ]:
DECK_PATH = DATA_DIR / "deck.csv"

if DECK_PATH.exists():
    deck = [int(line) for line in DECK_PATH.read_text().splitlines() if line.strip()]
    assert len(deck) == 60, f"Expected 60 cards, got {len(deck)}"
    print(f"Loaded {DECK_PATH} - {len(deck)} cards, {len(set(deck))} unique ids")
else:
    print("No deck yet. Export your Starmie or Festival list to data/deck.csv before holdout/submission.")


## 3. What each source contributes

### Doc 4/8 — Dragapult (`DragapultPolicy`)

Extracted into `build_merged_agent.py` -> `merged_agent_main.py`:

- `pre_turn_log` / `current_turn_log` turn tracking
- `set_card_counts()` deck reconstruction + prize inference
- `main_option_proc()` Phantom Dive combo planning
- Full contextual scoring loop inside `DragapultPolicy.choose()`

### Doc 10 — Expectimax (search layer only)

Extracted and wired to **`DragapultPolicy`**, not `AdvancedPolicy`:

- `_opponent_is_water_deck()` — Kyogre / Snover / Mega Abomasnow signals
- `_opponent_is_crustle_wall()` — Crustle stall detection
- `search_begin` / `search_step` rollout via `simulate_action()` + `rollout_turn()`
- `SEARCH_ALGO()` — Phase 1 exploration, Phase 2 **UCB1** candidate selection

> Expectimax's `AdvancedPolicy` is a Lucario heuristic scorer (`AttackPlan` + card-id tables). We deliberately **skip** it and keep Dragapult as the base policy.

### Docs 9/11 — Meta snapshot (manual steps in this notebook)

- Section 2 deck framing and candidate table
- Section 5 holdout gate scaffold mirroring `holdout_pass` / `PROMOTE_CANDIDATE` language from the snapshot notebook


## 4. Build merged `main.py`

`build_merged_agent.py` reads **Doc 4/8 + Doc 10** only, merges them, and writes `notebooks/merged_agent_main.py`. Copy that to repo-root `main.py` for submission.


In [ ]:
import subprocess
import sys

builder = NOTEBOOKS_DIR / "build_merged_agent.py"
subprocess.run([sys.executable, str(builder)], check=True)

merged_path = NOTEBOOKS_DIR / "merged_agent_main.py"
main_src = merged_path.read_text()
(REPO_ROOT / "main.py").write_text(main_src)
print(f"Wrote {REPO_ROOT / 'main.py'} ({len(main_src.splitlines())} lines)")


In [ ]:
REQUIRED_MARKERS = {
    "DragapultPolicy": "Doc 4/8 policy skeleton",
    "_opponent_is_water_deck": "Doc 10 opponent read",
    "_opponent_is_crustle_wall": "Doc 10 opponent read",
    "SEARCH_ALGO": "Doc 10 search wrapper",
    "Phase 2: UCB1": "Doc 10 UCB1 loop",
    "search_begin": "Doc 10 Search API",
}

main_text = (REPO_ROOT / "main.py").read_text()
checks = pd.DataFrame([
    {"marker": k, "purpose": v, "present": k in main_text}
    for k, v in REQUIRED_MARKERS.items()
])
missing = checks[~checks["present"]]["marker"].tolist()
print(checks.to_string(index=False))
if missing:
    raise RuntimeError(f"Build verification failed - missing: {missing}")
print("Build verification passed.")


## 5. Holdout validation (meta author gates)

Mirror the snapshot notebook's promotion language:

- `holdout_pass` + win rate >= threshold -> `PROMOTE_CANDIDATE`
- otherwise -> `HOLD_DO_NOT_SUBMIT`

Benchmark against ladder pillars **before** spending daily Kaggle submissions.


In [ ]:
HOLDOUT_OPPONENTS = [
    "starmie",
    "festival_thwackey",
    "hop_trevenant",
    "archaludon",
    "lucario",
]

HOLDOUT_GAMES = 40
PROMOTE_THRESHOLD = 0.52


def run_holdout_suite(opponents=HOLDOUT_OPPONENTS, games=HOLDOUT_GAMES):
    """Stub - connect to cabt / kaggle-environments once data/cg is available."""
    raise NotImplementedError(
        "Wire this to local simulators. Return list[dict] with wins/losses/ties per opponent."
    )


def summarize_holdout(results):
    rows = []
    for row in results:
        total = row["wins"] + row["losses"] + row["ties"]
        rate = row["wins"] / total if total else 0.0
        passed = rate >= PROMOTE_THRESHOLD
        rows.append({
            **row,
            "win_rate": rate,
            "holdout_gate": "holdout_pass" if passed else "holdout_fail",
            "verdict": "PROMOTE_CANDIDATE" if passed else "HOLD_DO_NOT_SUBMIT",
        })
    return pd.DataFrame(rows)


print("Stress pool:", HOLDOUT_OPPONENTS)
print("Games/opponent:", HOLDOUT_GAMES)
print("Promotion threshold:", PROMOTE_THRESHOLD)


## 6. Package Kaggle submission

Tarball must contain `main.py`, `deck.csv`, and `cg/` at the **top level**.


In [ ]:
import glob
import shutil
import tarfile


def find_cg_dir():
    patterns = [
        str(DATA_DIR / "cg"),
        str(REPO_ROOT / "cg"),
        "/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg",
        "/kaggle/input/**/sample_submission/cg",
        "/kaggle/input/**/cg-lib/cg",
    ]
    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            p = Path(path)
            if p.is_dir() and (p / "api.py").exists():
                return p
    raise FileNotFoundError("Place the cg SDK under data/cg before packaging.")


def build_submission(output=REPO_ROOT / "submission.tar.gz"):
    if not DECK_PATH.exists():
        raise FileNotFoundError(f"Missing deck: {DECK_PATH}")
    if not (REPO_ROOT / "main.py").exists():
        raise FileNotFoundError("Run section 4 first to generate main.py")

    cg_src = find_cg_dir()
    staging = REPO_ROOT / ".submission_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir()

    shutil.copy2(REPO_ROOT / "main.py", staging / "main.py")
    shutil.copy2(DECK_PATH, staging / "deck.csv")
    shutil.copytree(cg_src, staging / "cg")

    with tarfile.open(output, "w:gz") as tar:
        tar.add(staging / "main.py", arcname="main.py")
        tar.add(staging / "deck.csv", arcname="deck.csv")
        for item in sorted((staging / "cg").rglob("*")):
            if item.is_file() and "__pycache__" not in item.parts:
                tar.add(item, arcname=str(Path("cg") / item.relative_to(staging / "cg")))

    shutil.rmtree(staging)
    print(f"Created {output} ({output.stat().st_size / 1024 / 1024:.2f} MiB)")


# build_submission()  # uncomment after data/deck.csv and data/cg exist


## 7. End-to-end checklist

1. **Section 1** - confirm all three reference notebooks are present locally.
2. **Section 2** - choose Starmie or Festival; write `data/deck.csv`.
3. **Section 4** - build + verify merged agent (Dragapult skeleton + Expectimax search).
4. **Section 5** - run holdout; require `PROMOTE_CANDIDATE` before submitting.
5. **Section 6** - package `submission.tar.gz` and upload to Kaggle.

Keep reference notebooks under `docs/resources/` (gitignored). Local extraction scripts, if any, belong in `data/extractions/`.
